# Endless Racer -- Control Algorithms (Solution)

This is the **solution** notebook for the Endless Racer simulation
(`simulation.py`), which uses the custom Gymnasium-style `EndlessRacerEnv`
environment (`racer_env.py`): a top-down, vertically scrolling endless racer
with random wheel slippage and random drift, so the car never drives in a
straight line without external control.

**Environment recap**

| | value | meaning |
|---|---|---|
| Observation (obstacles off) | `[theta]` | car heading angle (rad, 0 = straight ahead, positive = right) |
| Observation (obstacles on) | `[theta, angle_1, ..., angle_5]` | heading + bearings to visible obstacles, nearest first; unused slots = `NO_OBSTACLE_ANGLE` (= pi) |
| Action | `Box(-1.0, 1.0, (1,))` | steering command (negative = left, positive = right) |
| Reward | forward distance per step | **-100** on collision with the lane edges or an obstacle (episode ends) |
| `info` dict | `lateral_offset`, `heading`, `distance`, `grip`, ... | extra quantities for control |

Each section below explains one control algorithm and contains a fully
worked implementation. The export cell at the bottom writes the code to
`../controllers.py`, the file used by `simulation.py`.


In [ ]:
import math
import os

import sys
sys.path.append("..")  # so we can import from the project root

import numpy as np

from racer_env import MAX_HEADING, NO_OBSTACLE_ANGLE

## 1. Manual Control

The simplest "controller" is you! In `simulation.py`, on every simulation
step the GUI checks which arrow keys are currently held down and passes that
information to `get_manual_action` as a dictionary, e.g.:

```python
pressed_keys = {"left": True, "right": False, "reset": False, "quit": False}
```

Your job is to turn this into a steering command for the car:

- If `"left"` is held, steer in the negative direction.
- If `"right"` is held, steer in the positive direction.
- If both or neither are held, apply zero net steering.
- The result must be a NumPy array with the action space's dtype, clipped to
  the action space bounds (`np.clip(action, action_space.low, action_space.high)`).

Remember the wheels randomly *slip*: sometimes your steering will barely do
anything for a second or two. That's the environment, not your code!

In [ ]:
def get_manual_action(action_space, pressed_keys, magnitude=1.0):
    """Convert currently-held keyboard keys into a steering action.

    Parameters
    ----------
    action_space : gymnasium.spaces.Box
        The environment's action space.
    pressed_keys : dict
        Maps key names (``"left"``, ``"right"``) to booleans indicating
        whether the corresponding arrow key is currently held down.
    magnitude : float
        Magnitude of the steering command to apply while a key is held.

    Returns
    -------
    numpy.ndarray
        The action to send to ``env.step``, clipped to ``action_space``.
    """
    steer = 0.0
    if pressed_keys.get("left"):
        steer -= magnitude
    if pressed_keys.get("right"):
        steer += magnitude

    action = np.array([steer], dtype=action_space.dtype)
    return np.clip(action, action_space.low, action_space.high)

## 2. Desired Heading (the PID setpoint)

A PID controller needs a *setpoint* -- the value it should drive the
measurement to. Here the measurement is the car's heading angle, and the
setpoint is the heading the car *should* have right now, computed by
`compute_desired_heading` from the observation and the `info` dict.

**Obstacles off.** The goal is to keep the car driving forward in the middle
of the lane. If the car has drifted to the right of the centre
(`info["lateral_offset"] > 0`), the desired heading should point slightly
left, and vice versa. The simplest scheme is proportional:

$$\theta_{desired} = -K_{center} \cdot x_{offset}$$

**Obstacles on.** Raw bearings are tricky to act on directly: a far-away car
that is laterally well clear of us can have a *smaller* bearing than a
nearby car we are about to scrape past, so a fixed "danger cone" on the
angles keeps misfiring. Instead we *plan a lateral target position* and
convert the lateral error into a heading. The observation gives us the
bearings; the `info` dict additionally gives the same cars as relative
`(dx, dy)` positions (`info["obstacles"]`, nearest first) -- the classical
controller is allowed richer telemetry than the RL agent, just as it
already uses `info["lateral_offset"]` in obstacles-off mode. The plan has
three ingredients:

1. **Reachability.** We can only swerve so fast, so the lane line of a car
   that is already close can no longer be crossed -- we must stay on
   whichever side of it we are now. Treat a line as crossable only while
   `dy > cross_lead + cross_per_m * |dx|`.
2. **Free gaps.** Every car within `plan_horizon` blocks the band of lateral
   positions within `clearance` of its centre. Whatever remains of the
   drivable corridor (`±corridor`) *and* is still reachable forms the free
   gaps. Aim for the gap that needs the smallest lateral move, drifting
   slightly towards its middle for margin.
3. **Squeeze fallback.** If no gap is free, put as much room as possible
   between us and the nearby traffic: drive (at full urgency) to whichever
   reachable extreme is farthest from the closest cars.

Finally turn the lateral error into a heading with a proportional law,
saturated at `max_dodge` so manoeuvres stay controlled:

$$\theta_{desired} = \mathrm{clip}\bigl(K_{track} \cdot (x_{target} - x_{offset}),\ -\theta_{dodge},\ \theta_{dodge}\bigr)$$

**Implementation steps**
1. With obstacles off, return the clipped centring term
   `-center_gain * info["lateral_offset"]`.
2. Convert `info["obstacles"]` into absolute lateral positions
   (`offset + dx`) for cars with `dy < plan_horizon`.
3. Compute the reachable range `[reach_lo, reach_hi]` by clamping at the
   lane line of every car that is no longer crossable.
4. Sweep the sorted blocked bands `[x - clearance, x + clearance]` to
   collect the free gaps inside the reachable range (discard slivers
   narrower than 0.5 m).
5. If there are free gaps, pick the one needing the smallest move and set
   the target inside it (blend slightly towards the gap middle). Otherwise
   squeeze: pick the reachable extreme farthest from all cars closer than
   15 m (use a higher gain for this emergency case).
6. Return `clip(gain * (target - offset))`, saturated at `max_dodge` and
   clipped to `[-MAX_HEADING, MAX_HEADING]`, as a `float`.

In [ ]:
def compute_desired_heading(
    observation,
    info,
    obstacles_on,
    center_gain=0.12,
    plan_horizon=25.0,
    cross_lead=7.0,
    cross_per_m=2.0,
    clearance=2.2,
    corridor=4.5,
    track_gain=0.25,
    max_dodge=0.40,
):
    """Compute the heading angle the car *should* have right now.

    Obstacles off
        The goal is simply to drive forward in the middle of the lane, so the
        desired heading steers gently back towards the lane centre:
        ``-center_gain * lateral_offset`` (if the car has drifted right, the
        desired heading points slightly left, and vice versa).

    Obstacles on
        The controller plans a safe *lateral target position* and converts the
        lateral error into a desired heading. The observation's obstacle
        bearings tell us where traffic is, and the ``info`` dict provides the
        same cars as relative ``(dx, dy)`` positions (the classical controller
        is allowed richer telemetry than the RL agent, just like it already
        uses ``lateral_offset`` in obstacles-off mode). The plan has three
        ingredients:

        1. *Reachability* -- we can only swerve so fast, so the lane line of a
           car that is already close cannot be crossed any more: we must stay
           on whichever side of it we currently are. A line is considered
           crossable only while ``dy > cross_lead + cross_per_m * |dx|``.
        2. *Free gaps* -- every car within ``plan_horizon`` blocks the band of
           lateral positions within ``clearance`` of its centre. Whatever is
           left of the drivable corridor (``+/- corridor``) and still
           reachable forms the free gaps. We aim for the gap that requires
           the smallest move, drifting slightly towards its middle for
           margin.
        3. *Squeeze fallback* -- if no gap is free, we put as much room as
           possible between us and nearby traffic by driving to whichever
           reachable extreme is farthest from the closest cars.

        The lateral error is finally turned into a heading via a simple
        proportional law, saturated at ``max_dodge`` so manoeuvres stay
        controlled.

    Parameters
    ----------
    observation : array-like
        The environment observation (see module docstring).
    info : dict
        The ``info`` dict from ``env.step``/``env.reset``; must contain
        ``"lateral_offset"`` and (in obstacles-on mode) ``"obstacles"``, a
        list of relative ``(dx, dy)`` positions of the visible traffic cars.
    obstacles_on : bool
        Which mode the environment is in.
    center_gain : float
        How strongly the car is steered back to the lane centre (rad per m).
    plan_horizon : float
        Cars further ahead than this (m) are ignored by the planner.
    cross_lead, cross_per_m : float
        Crossability rule: a car's lane line can only be crossed while
        ``dy > cross_lead + cross_per_m * |dx|``.
    clearance : float
        Half-width (m) of the lateral band each car blocks.
    corridor : float
        Half-width (m) of the drivable corridor the planner uses.
    track_gain : float
        Proportional gain (rad per m) from lateral error to desired heading.
    max_dodge : float
        Saturation (rad) of the avoidance heading.

    Returns
    -------
    float
        The desired heading angle (rad), clipped to ``+/- MAX_HEADING``.
    """
    offset = info["lateral_offset"]

    if not obstacles_on:
        return float(np.clip(-center_gain * offset, -MAX_HEADING, MAX_HEADING))

    # Traffic cars as absolute lateral position + distance ahead.
    cars = [
        (offset + dx, dy)
        for dx, dy in info.get("obstacles", [])
        if dy < plan_horizon
    ]

    # 1. Reachability: stay on our side of any line we can no longer cross.
    reach_lo, reach_hi = -corridor, corridor
    for car_x, dy in cars:
        crossable = dy > cross_lead + cross_per_m * abs(car_x - offset)
        if not crossable:
            if offset >= car_x:
                reach_lo = max(reach_lo, car_x)
            else:
                reach_hi = min(reach_hi, car_x)

    # 2. Free gaps between the blocked bands, inside the reachable range.
    free, lo = [], reach_lo
    for b_lo, b_hi in sorted((x - clearance, x + clearance) for x, _ in cars):
        if b_lo > lo:
            free.append((lo, min(b_lo, reach_hi)))
        lo = max(lo, b_hi)
    if lo < reach_hi:
        free.append((lo, reach_hi))
    free = [(a, b) for a, b in free if b - a > 0.5 and b > reach_lo]

    urgent = False
    if free:
        # Aim for the gap that needs the smallest lateral move.
        def move_needed(gap):
            g_lo, g_hi = gap
            if g_lo <= offset <= g_hi:
                return 0.0
            return min(abs(offset - g_lo), abs(offset - g_hi))

        g_lo, g_hi = min(free, key=move_needed)
        target = float(np.clip(offset, g_lo + 0.3, g_hi - 0.3))
        # Drift gently towards the middle of the gap for extra margin.
        target = 0.8 * target + 0.2 * (g_lo + g_hi) / 2.0
    else:
        # 3. Squeeze: maximise room to the closest cars, at full urgency.
        close = [c for c in cars if c[1] < 15.0]
        if close:
            def room(p):
                return min(abs(p - x) for x, _ in close)

            target = max((reach_lo, reach_hi), key=room)
            urgent = True
        else:
            target = float(np.clip(offset, reach_lo, reach_hi))

    gain = 0.5 if urgent else track_gain
    desired = np.clip(gain * (target - offset), -max_dodge, max_dodge)
    return float(np.clip(desired, -MAX_HEADING, MAX_HEADING))

## 3. PID Control

A **PID controller** computes a control signal from the *error* between the
desired setpoint and the current measurement:

$$ u(t) = K_p \, e(t) \;+\; K_i \int_0^t e(\tau)\, d\tau \;+\; K_d \frac{de(t)}{dt} $$

- **P (proportional)** -- reacts to the *current* error. Larger `Kp` means a
  stronger immediate correction, but too large causes oscillation.
- **I (integral)** -- accumulates *past* error over time, eliminating steady
  state bias. Too large causes overshoot/oscillation ("integral windup").
- **D (derivative)** -- reacts to *how fast* the error is changing, damping
  oscillations. Too large amplifies noise (and this environment is noisy!).

Here the measurement is the heading `theta = observation[0]`, the setpoint
is updated every step by `compute_desired_heading`, and the output is the
steering command.

**Implementation steps for `compute_action`**
1. Compute the error `e = setpoint - theta`.
2. Accumulate the integral: `integral += e * dt`.
3. Approximate the derivative: `(e - previous_e) / dt` (guard against
   `dt == 0`), then store `e` as the previous error.
4. Combine: `u = kp*e + ki*integral + kd*derivative`.
5. Return as a NumPy array with the action space's dtype, clipped to the
   action space bounds.

Also implement `reset` (clear the integral/previous error between episodes),
`set_gains` (called by the GUI sliders) and `set_setpoint`.

In [ ]:
class PIDController:
    """A PID controller that steers the car's heading to ``setpoint``.

    The controller drives the heading angle ``theta`` (``observation[0]``)
    to the desired heading by issuing a steering command:

        u(t) = Kp * e(t) + Ki * integral(e) + Kd * d(e)/dt

    where ``e(t) = setpoint(t) - theta(t)``. The setpoint is updated every
    step by ``compute_desired_heading`` (lane centring / obstacle avoidance).
    """

    def __init__(self, kp=0.0, ki=0.0, kd=0.0, setpoint=0.0):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.setpoint = setpoint
        self.reset()

    def reset(self):
        """Clear the integral and derivative memory (call between episodes)."""
        self._integral = 0.0
        self._prev_error = 0.0

    def set_gains(self, kp, ki, kd):
        """Update the P, I and D gains (e.g. from GUI sliders)."""
        self.kp = kp
        self.ki = ki
        self.kd = kd

    def set_setpoint(self, setpoint):
        """Update the desired heading angle (rad)."""
        self.setpoint = setpoint

    def compute_action(self, observation, action_space, dt):
        """Compute the steering action for the current observation.

        Parameters
        ----------
        observation : array-like
            The environment observation; ``observation[0]`` is the heading.
        action_space : gymnasium.spaces.Box
            The environment's action space, used to clip the output.
        dt : float
            Time step between calls (``env.dt``), used for the integral and
            derivative terms.

        Returns
        -------
        numpy.ndarray
            The action to send to ``env.step``, clipped to ``action_space``.
        """
        theta = observation[0]
        error = self.setpoint - theta

        self._integral += error * dt
        derivative = (error - self._prev_error) / dt if dt > 0 else 0.0
        self._prev_error = error

        output = self.kp * error + self.ki * self._integral + self.kd * derivative

        action = np.array([output], dtype=action_space.dtype)
        return np.clip(action, action_space.low, action_space.high)

## 4. Reinforcement Learning Control (PPO via Stable-Baselines3)

[Stable-Baselines3](https://stable-baselines3.readthedocs.io/) provides
ready-made implementations of popular RL algorithms. We'll use **PPO**
(Proximal Policy Optimization), a policy-gradient algorithm that works well
on continuous-control tasks like this one.

At a high level, PPO repeatedly:
1. Runs the current policy in the environment to collect a batch of
   `(state, action, reward)` experience (`n_steps` steps).
2. Computes advantage estimates (how much better an action was than average),
   controlled by `gamma` (discount factor) and `gae_lambda`.
3. Updates the policy network with several epochs of mini-batch gradient
   descent (`batch_size`), nudged by `learning_rate`, while *clipping* the
   policy update so it doesn't change too drastically (the "proximal" part).
4. `ent_coef` adds an entropy bonus to encourage exploration.

This repeats until `total_timesteps` environment steps have been collected.
The simulation's GUI lets the user edit all of these hyperparameters before
training starts. Note that the agent only observes *angles* -- in the
obstacles-on mode it never sees obstacle distances directly -- which makes
this a partially observable and genuinely challenging problem.

**Implementation steps for `train_rl_agent`**
1. Import `PPO` from `stable_baselines3`.
2. Construct `PPO("MlpPolicy", env, ...)`, passing through the hyperparameters
   from the `hyperparams` dict using `.get(key, default)` (so missing keys
   fall back to sensible defaults), plus `verbose=1`.
3. Call `model.learn(total_timesteps=total_timesteps, callback=callback)`.
4. Make sure the directory containing `save_path` exists
   (`os.makedirs(..., exist_ok=True)`), then `model.save(save_path)`.
5. Return the model.

`load_rl_agent` should call `PPO.load(path, env=env)`, and `get_rl_action`
should call `model.predict(observation, deterministic=deterministic)` and
return just the action.

In [ ]:
def train_rl_agent(env, hyperparams, total_timesteps, save_path, callback=None):
    """Train a PPO agent on ``env`` and save it to ``save_path``.

    Parameters
    ----------
    env : gymnasium.Env
        The (non-rendered) training environment.
    hyperparams : dict
        PPO hyperparameters. Recognised keys: ``learning_rate``, ``n_steps``,
        ``batch_size``, ``gamma``, ``gae_lambda``, ``ent_coef``. Missing keys
        fall back to Stable-Baselines3 defaults.
    total_timesteps : int
        Number of environment steps to train for.
    save_path : str
        Path (without/with ``.zip``) to save the trained model checkpoint.
    callback : stable_baselines3.common.callbacks.BaseCallback, optional
        Callback passed through to ``model.learn`` (e.g. for progress
        reporting in the GUI).

    Returns
    -------
    stable_baselines3.PPO
        The trained model.
    """
    from stable_baselines3 import PPO

    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=hyperparams.get("learning_rate", 3e-4),
        n_steps=hyperparams.get("n_steps", 1024),
        batch_size=hyperparams.get("batch_size", 64),
        gamma=hyperparams.get("gamma", 0.99),
        gae_lambda=hyperparams.get("gae_lambda", 0.95),
        ent_coef=hyperparams.get("ent_coef", 0.0),
        verbose=1,
    )

    model.learn(total_timesteps=total_timesteps, callback=callback)

    save_dir = os.path.dirname(os.path.abspath(save_path))
    os.makedirs(save_dir, exist_ok=True)
    model.save(save_path)

    return model


def load_rl_agent(path, env=None):
    """Load a previously-trained PPO model from ``path``.

    Parameters
    ----------
    path : str
        Path to the saved model (``.zip`` checkpoint).
    env : gymnasium.Env, optional
        Environment to attach to the loaded model.

    Returns
    -------
    stable_baselines3.PPO
        The loaded model.
    """
    from stable_baselines3 import PPO

    return PPO.load(path, env=env)


def get_rl_action(model, observation, deterministic=True):
    """Get the action chosen by a trained RL model for ``observation``.

    Parameters
    ----------
    model : stable_baselines3.PPO
        A trained (or loaded) model.
    observation : array-like
        The environment observation.
    deterministic : bool
        Whether to use the deterministic policy (recommended for evaluation).

    Returns
    -------
    numpy.ndarray
        The action to send to ``env.step``.
    """
    action, _ = model.predict(observation, deterministic=deterministic)
    return action

## Export to `controllers.py`

Running the cell below pulls the source code of each function/class defined
above out of the notebook and writes it to `../controllers.py`, the file
used by `simulation.py`. After exporting, test your implementation with:

```bash
python simulation.py
```

In [ ]:
import ast
import os

_OUTPUT_PATH = os.path.join("..", "controllers.py")

_HEADER = '''"""
Control algorithms for the EndlessRacerEnv environment (racer_env.py).

Auto-generated by exporting a notebook in notebooks/ -- re-run the export
cell in that notebook after making changes to regenerate this file.
"""

import math
import os

import numpy as np

from racer_env import MAX_HEADING, NO_OBSTACLE_ANGLE
'''

# Names of the functions/classes to export, in the order they should appear
# in controllers.py.
_NAMES = [
    "get_manual_action",
    "compute_desired_heading",
    "PIDController",
    "train_rl_agent",
    "load_rl_agent",
    "get_rl_action",
]


def _find_definition(name):
    """Find the source of the most recently executed `def`/`class name`.

    `inspect.getsource` doesn't work reliably for classes defined in
    notebooks, so instead we search the IPython input history (`In`) for a
    top-level function/class definition called `name` and pull out just that
    definition using the `ast` module.
    """
    for cell_source in reversed(In):
        try:
            tree = ast.parse(cell_source)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                if node.name == name:
                    return ast.get_source_segment(cell_source, node)
    raise RuntimeError(f"Could not find a definition for `{name}`. Did you run that cell?")


with open(_OUTPUT_PATH, "w") as f:
    f.write(_HEADER)
    for _name in _NAMES:
        f.write("\n\n")
        f.write(_find_definition(_name))
        f.write("\n")

print(f"Exported {len(_NAMES)} definitions to {os.path.abspath(_OUTPUT_PATH)}")